In [ ]:
import stim
import numpy as np

from spiderwarp.csscode import CSSCode
from spiderwarp.utils import get_project_root, load_state_prep_circuit
from spiderwarp.stim_utils import steane_se_from_stim_state_prep, get_num_measurements, get_circuit_depth
from spiderwarp.path_cover import CoveredZXGraph
from spiderwarp.qubit_reuse import build_circuit_dag, apply_logical_qubit_merge_and_compress, dag_to_circuit, inject_qubit_reuse, AggressiveDepthAwareStrategy
from spiderwarp.fault_tolerance_verification import verify_ftsp, verify_fault_order_detect_or_correct, classify_flag_patterns_by_fault_order, plan_minimal_followup_by_flag, print_followup_flow

%load_ext autoreload
%autoreload 2

In [ ]:
code_dir, code_name, circ_dir, circ_path = "misc", "32_20_4", "misc", "t1_zero_32_20_4"
code = CSSCode.load_code(code_dir, code_name)

In [ ]:
circuit = load_state_prep_circuit(circ_dir, circ_path)
se = steane_se_from_stim_state_prep(circuit, se_basis="Z", n=code.n)
covered = CoveredZXGraph.from_stim(se)
covered.basic_FE_rewrites()

print("Depth", get_circuit_depth(se))
print("Number of ancilla qubits necessary:", se.num_qubits - code.n)

In [ ]:
dag = build_circuit_dag(covered)

mod_dag, _, _ = inject_qubit_reuse(dag, code.n, AggressiveDepthAwareStrategy())
compressed_dag = apply_logical_qubit_merge_and_compress(mod_dag, code.n)
final_circ, final_meas_map = dag_to_circuit(compressed_dag)

print("Number of ancilla qubits necessary:", final_circ.num_qubits - code.n)
print("Mapping of new measurements to old measurements:", final_meas_map)


In [ ]:
# Build one Stim circuit whose detector samples are:
#   columns 0..num_flags-1: flag outcomes
#   remaining columns:      H_z syndrome bits
# Detector coordinates use (kind, index), where kind=0 is a flag and kind=1 is a syndrome.
num_final_measurements = final_circ.num_measurements
num_flags = get_num_measurements(se) - code.n

# final_meas_map is new measurement index -> original measurement ID.
# Original IDs [0, num_flags) are flags; the following code.n IDs are
# the data-qubit measurements in physical-qubit order.
expected_original_ids = set(range(num_flags + code.n))
assert len(final_meas_map) == num_final_measurements
assert set(final_meas_map.values()) == expected_original_ids
original_to_final = {original: final for final, original in final_meas_map.items()}

flag_measurements = [original_to_final[flag_id] for flag_id in range(num_flags)]
data_measurements = [
    original_to_final[num_flags + qubit] for qubit in range(code.n)
]

flagged_syndrome_circuit = final_circ.copy()

# A flag detector is just the corresponding flag measurement.
for flag_id, measurement_index in enumerate(flag_measurements):
    flagged_syndrome_circuit.append(
        "DETECTOR",
        [stim.target_rec(measurement_index - num_final_measurements)],
        [0, flag_id],
    )

# Each syndrome detector is the H_z-row parity of the 32 data measurements.
for syndrome_id, row in enumerate(code.H_z):
    record_targets = [
        stim.target_rec(data_measurements[qubit] - num_final_measurements)
        for qubit in np.flatnonzero(row)
    ]
    flagged_syndrome_circuit.append(
        "DETECTOR", record_targets, [1, syndrome_id]
    )

asset_path = get_project_root() / "assets/circuits/SECircuits/32_20_4.stim"
flagged_syndrome_circuit.to_file(asset_path)


In [ ]:
H_x, H_z, L_x, L_z, d = code.H_x, code.H_z, code.L_x, code.L_z, code.d
# A distance-d code corrects floor((d - 1) / 2) arbitrary faults.
t = (d - 1) // 2

# R prepares a physical all-zero state, not an encoded logical all-zero state.
# Build the unique encoded |0...0> state by fixing all X/Z stabilizers and
# every logical Z operator to eigenvalue +1. This prefix remains noiseless:
# the verifier inserts faults only into the syndrome-extraction gadget.
def pauli_from_row(row, pauli):
    return stim.PauliString("".join(pauli if bit else "_" for bit in row))

encoded_zero_stabilizers = (
    [pauli_from_row(row, "X") for row in H_x]
    + [pauli_from_row(row, "Z") for row in H_z]
    + [pauli_from_row(row, "Z") for row in np.atleast_2d(L_z)]
)
ideal_encoded_zero = stim.Tableau.from_stabilizers(
    encoded_zero_stabilizers,
    allow_redundant=False,
    allow_underconstrained=False,
).to_circuit(method="elimination")


# The saved flag and syndrome detectors must all be quiet without faults.
noiseless_samples = (
    ideal_encoded_zero + flagged_syndrome_circuit
).compile_detector_sampler().sample(100)
assert not noiseless_samples.any()

print("\nRunning ideal-input syndrome-extraction FT verification...")
ft_result = verify_ftsp(
    prep_circ=flagged_syndrome_circuit,
    ideal_input_circuit=ideal_encoded_zero,
    H_primary=H_z, L_primary=np.atleast_2d(L_z),
    H_conjugate=H_x,
    d=d, t=t,
    primary_basis="Z", conjugate_basis="X",
    verbose=True,
)
print("Verification Result:", ft_result is True)

In [ ]:
# Verify one-fault correction plus two-fault detection-or-correction.
# A two-fault symptom may reuse a one-fault correction only when both
# histories have the same residual stabilizer coset. Otherwise it must have
# a new flag+syndrome symptom, allowing the state to be discarded.
correctable_faults = 1
detectable_faults = 2
assert correctable_faults + detectable_faults == d - 1

print("Syndrome-extraction one-correct/two-detect-or-correct:")
se_fault_order = verify_fault_order_detect_or_correct(
    circuit=flagged_syndrome_circuit,
    H_x=H_x, H_z=H_z, L_x=L_x, L_z=L_z,
    correctable_faults=correctable_faults,
    detectable_faults=detectable_faults,
    flag_detector_indices=range(num_flags),
    ideal_input_circuit=ideal_encoded_zero,
    prepared_state="0",
    verbose=True,
)
print("Result:", se_fault_order is True)
if se_fault_order is not True:
    display(se_fault_order.diagram('timeline-svg'))

se_flag_orders = classify_flag_patterns_by_fault_order(
    circuit=flagged_syndrome_circuit,
    H_x=H_x, H_z=H_z, L_x=L_x, L_z=L_z,
    flag_detector_indices=range(num_flags),
    ideal_input_circuit=ideal_encoded_zero,
    prepared_state="0",
)

se_followup_flow = plan_minimal_followup_by_flag(
    se_flag_orders, H_x, H_z, L_x, L_z, prepared_state="0"
)
print("\nMinimal SE follow-up flow:")
print_followup_flow(se_followup_flow, explain_corrections=False)
# Set explain_corrections=True to show each immediate correction and
# the resulting effective final states and weights.